# Image Classification Training

Train a custom image classifier on your Roboflow dataset (folder structure).

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

**After running the install cell, RESTART the runtime** (Runtime > Restart session), then run all cells from the top.

## 1. Download Dataset from Roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("folder")

## 2. Install Dependencies

**After this cell finishes, RESTART the runtime (Runtime > Restart session) then run all cells from the top again.**

In [ ]:
try:
    import setuptools
    sv = int(setuptools.__version__.split('.')[0])
    assert 70 <= sv < 80, f"setuptools must be 70<=v<80, got {setuptools.__version__}"
    import pkg_resources
    import numpy as np
    assert np.__version__.startswith("1."), f"numpy must be <2, got {np.__version__}"
    import torch
    assert torch.__version__.startswith("2.3"), f"torch must be 2.3.x, got {torch.__version__}"
    import timm, openvino, nncf, onnx
    from packaging import version as _v
    assert _v.parse(timm.__version__) >= _v.parse("1.0.3"), f"timm must be >=1.0.3, got {timm.__version__}"
    print(f"Already installed:")
    print(f"  setuptools: {setuptools.__version__}")
    print(f"  numpy:      {np.__version__}")
    print(f"  torch:      {torch.__version__}")
    print(f"  timm:       {timm.__version__}")
    print(f"  openvino:   {openvino.__version__}")
    print(f"  nncf:       {nncf.__version__}")
except (ImportError, AssertionError, ModuleNotFoundError):
    print("Installing (first time, ~2-3 min)...")

    !rm -rf /usr/lib/python3/dist-packages/pkg_resources
    !pip uninstall -y torch torchvision torchaudio openvino nncf openxlab --quiet
    !pip install torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121 --quiet
    !pip install openvino nncf onnx --quiet
    !pip install -U "timm>=1.0.3" --quiet

    !pip uninstall -y openxlab --quiet
    !pip install --force-reinstall "setuptools>=70,<80" --quiet
    !pip install "numpy<2" --quiet

    print("\n>>> RESTART RUNTIME and run all cells from the top <<<")

## 3. Training Parameters

In [ ]:
import os

# ============================================================
# EDIT THIS SECTION
# ============================================================
EPOCHS     = 50
BATCH_SIZE = 64
IMAGE_SIZE = (224, 224)
LR         = 0.02

# OPTIMIZE = True   -> produces a smaller, faster model (recommended for edge deployment)
#                      Falls back automatically if the optimization is unstable.
# OPTIMIZE = False  -> skip optimization, keep the standard model.
OPTIMIZE = True

# ============================================================
# AUGMENTATION -- verification-station preset. Set any value to 0
# / 0.0 to disable that op.
#
# For verification tasks with a fixed camera and a known ROI, the
# augmentations that pay off are the ones that mirror real-world
# variance at the station:
#   - lighting drift (brightness / contrast)
#   - reflections and specular highlights (saturation / hue)
#   - small camera or robot repeatability jitter (translate)
#   - occasional partial occlusion (erasing)
# `rotation` = 5 tolerates small mount/repeatability shifts without
# teaching the model to accept upside-down poses.
#
# Flips are OFF by default: on a workstation the object orientation
# is fixed, and a horizontal flip trains the model to accept a state
# that will never occur (helping neither pass nor fail).
# ============================================================
AUG = {
    "horizontal_flip": 0.0,   # probability; leave 0 for fixed-orientation stations
    "vertical_flip":   0.0,   # probability; leave 0 for fixed-orientation stations
    "rotation":        5,     # max degrees; tolerates small robot repeatability jitter
    "translate":       0.05,  # max fraction of image size; tolerates small ROI drift
    "shear":           0,     # max degrees; 0 disables
    "brightness":      0.2,   # jitter strength; lighting drift over a day
    "contrast":        0.2,   # jitter strength; lighting drift over a day
    "saturation":      0.2,   # jitter strength; reflections on caps/vials
    "hue":             0.05,  # jitter strength; small color-cast shifts
    "blur":            0.0,   # probability of Gaussian blur; enable if focus varies
    "erasing":         0.1,   # probability of random erasing; tolerates partial occlusion
}

# ============================================================
# BACKBONE
# `mobilenetv4_conv_small` is the right default for verification
# with a known small ROI: ~30 ms on CPU (int8), and small ROIs
# don't need more capacity -- larger backbones tend to overfit
# on limited per-class data. Bump only if the data variance is
# genuinely too rich for MobileNet to capture:
#   mobilenetv4_conv_medium  : +2-4% typical accuracy, ~50 ms
#   convnext_tiny            : +4-6% typical, ~110 ms
# ============================================================
BACKBONE = "mobilenetv4_conv_small"
project_name = project.name.lower().replace(" ", "_")
classes = list(project.classes.keys())

train_path = dataset.location + "/train"
valid_path = dataset.location + "/valid"

print(f"Classes:    {classes}")
print(f"Epochs:     {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Optimize:   {OPTIMIZE}")

## 4. Train

In [ ]:
import time
import copy
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
from timm.utils import ModelEmaV2

LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS   = min(5, max(1, EPOCHS // 10))


class LetterboxTransform:
    def __init__(self, target_size=(224, 224), color=128):
        self.target_size = target_size
        self.color = color

    def __call__(self, image):
        arr = np.array(image)
        ih, iw = arr.shape[:2]
        tw, th = self.target_size
        scale = min(tw / iw, th / ih)
        nw, nh = int(iw * scale), int(ih * scale)
        resized = cv2.resize(arr, (nw, nh), interpolation=cv2.INTER_LINEAR)
        out = np.full((th, tw, 3), self.color, dtype=np.uint8)
        pw = (tw - nw) // 2
        ph = (th - nh) // 2
        out[ph:ph + nh, pw:pw + nw] = resized
        return Image.fromarray(out)


# ------------------------------------------------------------
# Build training transform from the AUG dict. Each knob is
# independent — set to 0 / 0.0 to disable that op.
# ------------------------------------------------------------
train_tx = [LetterboxTransform(target_size=IMAGE_SIZE, color=128)]

if AUG.get("horizontal_flip", 0) > 0:
    train_tx.append(transforms.RandomHorizontalFlip(p=float(AUG["horizontal_flip"])))
if AUG.get("vertical_flip", 0) > 0:
    train_tx.append(transforms.RandomVerticalFlip(p=float(AUG["vertical_flip"])))

_rot = float(AUG.get("rotation", 0))
_tr  = float(AUG.get("translate", 0))
_sh  = float(AUG.get("shear", 0))
if _rot > 0 or _tr > 0 or _sh > 0:
    train_tx.append(transforms.RandomAffine(
        degrees=_rot,
        translate=(_tr, _tr) if _tr > 0 else None,
        shear=_sh if _sh > 0 else 0,
        fill=128,
    ))

cj = {k: float(AUG.get(k, 0)) for k in ("brightness", "contrast", "saturation", "hue")}
if any(v > 0 for v in cj.values()):
    train_tx.append(transforms.ColorJitter(**cj))

if AUG.get("blur", 0) > 0:
    train_tx.append(transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))],
        p=float(AUG["blur"]),
    ))

train_tx.extend([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if AUG.get("erasing", 0) > 0:
    train_tx.append(transforms.RandomErasing(p=float(AUG["erasing"])))

train_transform = transforms.Compose(train_tx)

val_transform = transforms.Compose([
    LetterboxTransform(target_size=IMAGE_SIZE, color=128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

active_augs = [k for k, v in AUG.items() if (isinstance(v, (int, float)) and v > 0)]
print(f"Active augmentations: {active_augs if active_augs else 'none'}")

train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
val_dataset = datasets.ImageFolder(valid_path, transform=val_transform) if os.path.isdir(valid_path) else None

# Balanced sampling — inverse-frequency per class. No-op when classes are balanced.
labels_all = [lbl for _, lbl in train_dataset.samples]
class_counts = np.bincount(labels_all, minlength=len(train_dataset.classes))
sample_weights = 1.0 / class_counts[labels_all]
sampler = WeightedRandomSampler(
    weights=sample_weights.tolist(),
    num_samples=len(labels_all),
    replacement=True,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader = (DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
              if val_dataset else None)

num_classes = len(train_dataset.classes)
classes = train_dataset.classes
print(f"Classes ({num_classes}): {classes}")
print(f"Train samples: {len(train_dataset)}  (per-class: {class_counts.tolist()})")
if val_dataset:
    print(f"Val samples:   {len(val_dataset)}")

# Scale EMA decay to dataset size so EMA reaches steady state within ~20% of training.
# Small datasets need much lower decay than the ImageNet default (0.9999).
total_steps = max(1, len(train_loader) * EPOCHS)
target_lag  = max(50, total_steps // 5)
EMA_DECAY   = max(0.95, min(0.9999, 1.0 - 1.0 / target_lag))
print(f"EMA decay:     {EMA_DECAY:.4f}  (target lag {target_lag} steps over {total_steps} total)")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model(BACKBONE, pretrained=True, num_classes=num_classes)
model = model.to(device)
ema = ModelEmaV2(model, decay=EMA_DECAY, device=device)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)
warmup = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS))
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

os.makedirs("./work_dir", exist_ok=True)
best_ckpt = "./work_dir/best.pth"
best_val = -1.0
t_train = time.time()


def eval_loader(m, loader):
    m.eval()
    c = t = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = m(inputs)
            _, pred = outputs.max(1)
            t += labels.size(0)
            c += pred.eq(labels).sum().item()
    return 100 * c / max(t, 1)


for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = total = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        ema.update(model)
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    scheduler.step()
    train_acc = 100 * correct / max(total, 1)

    val_acc_raw = val_acc_ema = None
    if val_loader:
        val_acc_raw = eval_loader(model, val_loader)
        val_acc_ema = eval_loader(ema.module, val_loader)
        if val_acc_ema >= val_acc_raw:
            chosen_acc, chosen_state = val_acc_ema, ema.module.state_dict()
        else:
            chosen_acc, chosen_state = val_acc_raw, model.state_dict()
        if chosen_acc > best_val:
            best_val = chosen_acc
            torch.save(chosen_state, best_ckpt)

    msg = f"Epoch {epoch+1:3d}/{EPOCHS}  loss {running_loss/len(train_loader):.4f}  train_acc {train_acc:5.2f}%"
    if val_acc_raw is not None:
        msg += f"  val_raw {val_acc_raw:5.2f}%  val_ema {val_acc_ema:5.2f}%  (best {best_val:5.2f}%)"
    print(msg)

if not os.path.exists(best_ckpt):
    torch.save(model.state_dict(), best_ckpt)
    best_val = train_acc

model.load_state_dict(torch.load(best_ckpt, map_location="cpu"))
model = model.cpu().eval()
print(f"\nTraining done in {(time.time()-t_train):.0f}s. Best val acc: {best_val:.2f}%")

## 5. Export & Optimize Model

In [ ]:
import torch.onnx
import openvino as ov
import nncf

if not hasattr(torch.onnx, "_dynamo_patched"):
    _orig_export = torch.onnx.export
    def _patched_export(*args, **kwargs):
        kwargs.pop("dynamo", None)
        return _orig_export(*args, **kwargs)
    torch.onnx.export = _patched_export
    torch.onnx._dynamo_patched = True

os.makedirs("./export", exist_ok=True)
onnx_path = "./export/model.onnx"

dummy = torch.zeros(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1])
torch.onnx.export(
    model, dummy, onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["logits"],
)
print(f"Exported model: {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.1f} MB)")

core = ov.Core()
ov_model = core.read_model(onnx_path)
std_xml = "./export/model.xml"
ov.save_model(ov_model, std_xml)
std_bin = std_xml.replace(".xml", ".bin")
print(f"Standard model: {std_bin} ({os.path.getsize(std_bin)/1024/1024:.1f} MB)")

H, W = IMAGE_SIZE
MEAN_255 = np.array([0.485, 0.456, 0.406], dtype=np.float32) * 255
STD_255  = np.array([0.229, 0.224, 0.225], dtype=np.float32) * 255
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}


def letterbox_cv(img, size, color=128):
    ih, iw = img.shape[:2]
    tw, th = size
    scale = min(tw / iw, th / ih)
    nw, nh = int(iw * scale), int(ih * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    out = np.full((th, tw, 3), color, dtype=np.uint8)
    pw = (tw - nw) // 2
    ph = (th - nh) // 2
    out[ph:ph + nh, pw:pw + nw] = resized
    return out


def preprocess_file(path):
    img = cv2.imread(path)
    if img is None:
        return None
    lb = letterbox_cv(img, (W, H), 128)
    rgb = cv2.cvtColor(lb, cv2.COLOR_BGR2RGB).astype(np.float32)
    return ((rgb - MEAN_255) / STD_255).transpose(2, 0, 1)[None]


def gather_per_class(root, per_class_limit):
    out = []
    if not os.path.isdir(root):
        return out
    for cls in sorted(os.listdir(root)):
        d = os.path.join(root, cls)
        if not os.path.isdir(d):
            continue
        files = sorted(f for f in os.listdir(d) if os.path.splitext(f)[1].lower() in IMG_EXT)
        out.extend(os.path.join(d, f) for f in files[:per_class_limit])
    return out


opt_xml = None
opt_bin = None

if OPTIMIZE:
    print("\nOptimizing model...")
    calib_paths = gather_per_class(train_path, per_class_limit=30)
    calib_tensors = [t for t in (preprocess_file(p) for p in calib_paths) if t is not None]
    print(f"  Calibration samples: {len(calib_tensors)}")
    assert len(calib_tensors) > 0, "No calibration images found"

    ov_model_q = core.read_model(onnx_path)
    calib_dataset = nncf.Dataset(calib_tensors, lambda x: x)
    optimized = nncf.quantize(
        ov_model_q,
        calib_dataset,
        preset=nncf.QuantizationPreset.MIXED,
        subset_size=len(calib_tensors),
    )
    opt_xml_candidate = "./export/model_opt.xml"
    ov.save_model(optimized, opt_xml_candidate)
    opt_bin_candidate = opt_xml_candidate.replace(".xml", ".bin")
    print(f"  Optimized model: {opt_bin_candidate} ({os.path.getsize(opt_bin_candidate)/1024/1024:.1f} MB)")
    print(f"  Size reduction: {100*(1 - os.path.getsize(opt_bin_candidate)/os.path.getsize(std_bin)):.0f}%")

    compiled_std = core.compile_model(ov_model, "CPU")
    compiled_opt = core.compile_model(optimized, "CPU")
    check_root = valid_path if os.path.isdir(valid_path) else train_path
    check_paths = gather_per_class(check_root, per_class_limit=10)
    match = total_checked = 0
    for p in check_paths:
        t = preprocess_file(p)
        if t is None:
            continue
        fp = int(np.argmax(list(compiled_std([t]).values())[0]))
        i8 = int(np.argmax(list(compiled_opt([t]).values())[0]))
        total_checked += 1
        match += int(fp == i8)
    agr_pct = 100 * match / max(total_checked, 1)
    print(f"  Stability check: {agr_pct:.1f}% agreement ({total_checked} images)")

    if agr_pct >= 90.0:
        opt_xml = opt_xml_candidate
        opt_bin = opt_bin_candidate
        print("  Optimization OK — packaging optimized version")
    else:
        print("  Optimization unstable — packaging standard version only")
else:
    print(f"\nOPTIMIZE={OPTIMIZE} — packaging standard version only.")

## 6. Save & Download Pickle

In [ ]:
import pickle

if opt_xml and opt_bin and os.path.exists(opt_xml):
    final_xml_path = opt_xml
    final_bin_path = opt_bin
    precision_tag = "int8"
    variant = "optimized"
else:
    final_xml_path = std_xml
    final_bin_path = std_bin
    precision_tag = "fp32"
    variant = "standard"

with open(final_xml_path, "r", encoding="utf-8") as f:
    xml_data = f.read()
with open(final_bin_path, "rb") as f:
    bin_data = f.read()

try:
    colors = project.colors
    if not colors:
        raise AttributeError
except Exception:
    palette = ["#ff3333", "#33aaff", "#33cc66", "#ffcc00", "#aa33ff", "#ff8833", "#00c2a8", "#e85d75"]
    colors = {c: palette[i % len(palette)] for i, c in enumerate(classes)}

model_dict = {
    "bin": bin_data,
    "xml": xml_data,
    "cls": classes,
    "colors": colors,
    "meta": {
        "model": BACKBONE,
        "type": "cls",
        "image_size": list(IMAGE_SIZE),
        "precision": precision_tag,
    },
}

pickle_path = f"./{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Pickle contains {variant} model ({len(bin_data)/1024/1024:.1f} MB)")
print(f"\nSaved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path)/1024/1024:.1f} MB")
print(f"Classes: {classes}")
print(f"Image size: {IMAGE_SIZE}")

try:
    from google.colab import files
    files.download(pickle_path)
except ImportError:
    pass